# Hallucination Detection Pipeline
This notebook runs the full pipeline on Google Colab with a free T4 GPU:
1. **Setup** — Install dependencies, upload dataset, login to HuggingFace
2. **Extract** — Run LLM forward passes and save per-layer hidden state features
3. **Probe** — Train logistic regression probes on each layer
4. **Ablation** — Train/test probes across domains
5. **Geometric** — PCA visualization and distance analysis

**Before running:** Set runtime to GPU via `Runtime > Change runtime type > T4 GPU`

todos:

1. Address overfitting with the linear regression model
    1. First, start with different penalties (likely wouldn’t change anything) / Try Elastic Net?
    2. Ensemble classification w/ boosted linear regression?
2. Add MLP to check if there’s a non-linear signal between them
3. Add PCA and try linear regression that way
4. If none of these work, then we see that there’s likely no separability between logistic regression and what
5. If there is separability, then we want to see in particular, if hallucinations cluster at all… but hallucinations is a probabalistic thing, right? What’s a plan to check for hallucinations and see if they’re different from anything else



```
# This is formatted as code
```



todos:

1. Address overfitting with the linear regression model
    1. First, start with different penalties (likely wouldn’t change anything) / Try Elastic Net?
    2. Ensemble classification w/ boosted linear regression?
2. Add MLP to check if there’s a non-linear signal between them
3. Add PCA and try linear regression that way
4. If none of these work, then we see that there’s likely no separability between logistic regression and what
5. If there is separability, then we want to see in particular, if hallucinations cluster at all… but hallucinations is a probabalistic thing, right? What’s a plan to check for hallucinations and see if they’re different from

## Step 0: Setup

In [ ]:
!pip install -q torch transformers accelerate pandas pyarrow scikit-learn matplotlib huggingface_hub bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.4 MB/s eta 0:00:00


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
from huggingface_hub import login
login()  # paste your HuggingFace token when prompted

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from google.colab import files
print("Upload your dataset.parquet file:")
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATASET_PATH}")

Upload your dataset.parquet file:


Saving dataset.parquet to dataset.parquet
Uploaded: dataset.parquet


In [ ]:
# ---- CONFIG ----
MODEL_NAME = "meta-llama/Llama-2-13b-hf"   # change to any HF model
FEATURES_DIR = "features"
BATCH_SIZE = 4
MAX_LENGTH = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROMPT_TEMPLATE = "Answer True/False and explain briefly: {statement}"
PROBE_C = 1.0
TEST_SIZE = 0.2
SEED = 42
N_PAIRS = 10000
COV_REG = 1e-4
USE_4BIT = False  # 4-bit quantization — required for 13B models on T4 (16 GB VRAM)

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"4-bit quantization: {USE_4BIT}")
print(f"Batch size: {BATCH_SIZE}")

Model: meta-llama/Llama-2-13b-hf
Device: cuda
4-bit quantization: False
Batch size: 4


## Step 1: Extract Features
Run forward passes through the LLM and save hidden state activations for each layer.

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def _infer_input_device(model, fallback_device: str) -> torch.device:
    """ Just sets up CUDA shenanigans """
    if hasattr(model, "hf_device_map") and isinstance(model.hf_device_map, dict):
        for key in ["model.embed_tokens", "embed_tokens", "lm_head"]:
            if key in model.hf_device_map:
                dev = model.hf_device_map[key]
                if isinstance(dev, int):
                    return torch.device(f"cuda:{dev}")
                return torch.device(str(dev))
        first_dev = next(iter(model.hf_device_map.values()), None)
        if first_dev is not None:
            if isinstance(first_dev, int):
                return torch.device(f"cuda:{first_dev}")
            return torch.device(str(first_dev))
    return torch.device(fallback_device)


def extract_features(model_name, dataset_path, out_dir, batch_size, max_length, device, prompt_template, use_4bit=False):
    """ Reads data from dataset.parquet, """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    df = pd.read_parquet(dataset_path)
    required = {"id", "domain", "label", "statement"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {', '.join(sorted(missing))}")

    texts = [prompt_template.format(statement=str(s)) for s in df["statement"].tolist()]
    labels = df["label"].astype(int).to_numpy()
    domains = df["domain"].astype(str).to_numpy()
    ids = df["id"].astype(str).to_numpy()
    print(f"Loaded {len(texts)} examples")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    dtype = torch.float16 if device.startswith("cuda") else torch.float32

    if use_4bit and device.startswith("cuda"):
        print("Loading model with 4-bit quantization (NF4)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, output_hidden_states=True, device_map="auto",
        )
    elif device.startswith("cuda"):
        print(f"Loading model in {dtype}...")
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, output_hidden_states=True, device_map="auto")
    else:
        print(f"Loading model in {dtype}...")
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, output_hidden_states=True)
        model.to(device)
    model.eval()
    input_device = _infer_input_device(model, fallback_device=device)
    print(f"Model loaded. Input device: {input_device}")

    sample = tokenizer("test", return_tensors="pt")
    sample = {k: v.to(input_device) for k, v in sample.items()}
    with torch.no_grad():
        out = model(**sample)
    num_layers = len(out.hidden_states)
    hidden_dim = out.hidden_states[0].shape[-1]
    print(f"Detected {num_layers} layers, hidden_dim={hidden_dim}")

    feats_by_layer = [[] for _ in range(num_layers)]
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for batch_num, i in enumerate(range(0, len(texts), batch_size)):
        if batch_num % 20 == 0:
            print(f"  Batch {batch_num + 1}/{total_batches}")
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
        enc = {k: v.to(input_device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc)
            hs = out.hidden_states

        attn = enc["attention_mask"]
        last_idx = (attn.sum(dim=1) - 1).clamp(min=0)
        batch_idx = torch.arange(last_idx.shape[0], device=last_idx.device)

        for layer_idx in range(num_layers):
            h_last = hs[layer_idx][batch_idx, last_idx, :]
            feats_by_layer[layer_idx].append(h_last.detach().float().cpu().numpy())

    print("Saving features...")
    meta = {
        "model_name": model_name, "num_layers": num_layers, "hidden_dim": hidden_dim,
        "n_examples": len(texts), "max_length": max_length, "batch_size": batch_size,
        "device": device, "prompt_template": prompt_template,
        "layer_0_is_embedding_output": True, "feature_dtype": "float32",
    }
    with (out_path / "meta.json").open("w") as f:
        json.dump(meta, f, indent=2)

    np.savez_compressed(out_path / "labels_domains_ids.npz", y=labels, domain=domains, id=ids)
    for layer_idx in range(num_layers):
        x = np.concatenate(feats_by_layer[layer_idx], axis=0)
        np.savez_compressed(out_path / f"layer_{layer_idx:02d}.npz", X=x)
        print(f"  Saved layer_{layer_idx:02d}.npz  shape={x.shape}")

    print(f"\nExtraction complete. {num_layers} layers saved to {out_dir}/")
    return num_layers, hidden_dim

In [ ]:
num_layers, hidden_dim = extract_features(
    model_name=MODEL_NAME,
    dataset_path=DATASET_PATH,
    out_dir=FEATURES_DIR,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE,
    prompt_template=PROMPT_TEMPLATE,
    use_4bit=USE_4BIT,
)

Loaded 9427 examples


config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Model loaded. Input device: cuda:0
Detected 41 layers, hidden_dim=5120
  Batch 1/2357
  Batch 21/2357
  Batch 41/2357
  Batch 61/2357
  Batch 81/2357
  Batch 101/2357
  Batch 121/2357
  Batch 141/2357
  Batch 161/2357
  Batch 181/2357
  Batch 201/2357
  Batch 221/2357
  Batch 241/2357
  Batch 261/2357
  Batch 281/2357
  Batch 301/2357
  Batch 321/2357
  Batch 341/2357
  Batch 361/2357
  Batch 381/2357
  Batch 401/2357
  Batch 421/2357
  Batch 441/2357
  Batch 461/2357
  Batch 481/2357
  Batch 501/2357
  Batch 521/2357
  Batch 541/2357
  Batch 561/2357
  Batch 581/2357
  Batch 601/2357
  Batch 621/2357
  Batch 641/2357
  Batch 661/2357
  Batch 681/2357
  Batch 701/2357
  Batch 721/2357
  Batch 741/2357
  Batch 761/2357
  Batch 781/2357
  Batch 801/2357
  Batch 821/2357
  Batch 841/2357
  Batch 861/2357
  Batch 881/2357
  Batch 901/2357
  Batch 921/2357
  Batch 941/2357
  Batch 961/2357
  Batch 981/2357
  Batch 1001/2357
  Batch 1021/2357
  Batch 1041/2357
  Batch 1061/2357
  Batch 1081/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
!cp -r "/content/features" "/content/drive/MyDrive/features"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2: Layer Sweep (Probe Training)
Train a logistic regression probe on each layer and evaluate accuracy + AUROC.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score


def train_logreg_probe(X_train, y_train, C=1.0):
    clf = LogisticRegression(penalty="l2", C=C, solver="liblinear", max_iter=2000, class_weight="balanced")
    clf.fit(X_train, y_train)
    return clf


def eval_binary_classifier(clf, X, y):
    probs = clf.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    metrics = {"acc": float(accuracy_score(y, preds))}
    try:
        metrics["auroc"] = float(roc_auc_score(y, probs))
    except ValueError:
        metrics["auroc"] = float("nan")
    return metrics


def build_train_test_indices(n, test_size, seed):
    rng = np.random.RandomState(seed)
    shuffled = np.arange(n)
    rng.shuffle(shuffled)
    n_test = max(1, min(n - 1, int(round(n * test_size))))
    return shuffled[n_test:], shuffled[:n_test]


def run_layer_sweep(features_dir, test_size, seed, C):
    features_path = Path(features_dir)
    shared = np.load(features_path / "labels_domains_ids.npz", allow_pickle=True)
    y = shared["y"]

    train_idx, test_idx = build_train_test_indices(len(y), test_size, seed)

    layer_files = sorted(features_path.glob("layer_*.npz"), key=lambda p: int(p.stem.split("_")[1]))
    print(f"Found {len(layer_files)} layer files, n_train={len(train_idx)}, n_test={len(test_idx)}")

    rows = []
    for layer_file in layer_files:
        layer_idx = int(layer_file.stem.split("_")[1])
        X = np.load(layer_file)["X"]

        clf = train_logreg_probe(X[train_idx], y[train_idx], C=C)
        train_m = eval_binary_classifier(clf, X[train_idx], y[train_idx])
        test_m = eval_binary_classifier(clf, X[test_idx], y[test_idx])

        row = {
            "layer": layer_idx, "n_train": len(train_idx), "n_test": len(test_idx),
            "train_acc": train_m["acc"], "train_auroc": train_m["auroc"],
            "test_acc": test_m["acc"], "test_auroc": test_m["auroc"],
        }
        rows.append(row)
        print(f"  layer={layer_idx:02d}  train_acc={row['train_acc']:.4f}  train_auroc={row['train_auroc']:.4f}  test_acc={row['test_acc']:.4f}  test_auroc={row['test_auroc']:.4f}")

    return pd.DataFrame(rows).sort_values("layer").reset_index(drop=True)

In [ ]:
sweep_df = run_layer_sweep(FEATURES_DIR, test_size=TEST_SIZE, seed=SEED, C=PROBE_C)

best_row = sweep_df.loc[sweep_df["test_auroc"].idxmax()]
BEST_LAYER = int(best_row["layer"])
print(f"\nBest layer: {BEST_LAYER} (test_auroc={best_row['test_auroc']:.4f})")

sweep_df.to_csv("layer_sweep_results.csv", index=False)
print("Saved layer_sweep_results.csv")
sweep_df

Found 41 layer files, n_train=7542, n_test=1885
  layer=00  train_acc=0.6946  train_auroc=0.7623  test_acc=0.5496  test_auroc=0.5627
  layer=01  train_acc=0.7058  train_auroc=0.7916  test_acc=0.5570  test_auroc=0.5691
  layer=02  train_acc=0.7629  train_auroc=0.8613  test_acc=0.5639  test_auroc=0.5729
  layer=03  train_acc=0.8347  train_auroc=0.9267  test_acc=0.5602  test_auroc=0.5578
  layer=04  train_acc=0.8828  train_auroc=0.9598  test_acc=0.5698  test_auroc=0.5822
  layer=05  train_acc=0.9100  train_auroc=0.9737  test_acc=0.5814  test_auroc=0.5795
  layer=06  train_acc=0.9287  train_auroc=0.9850  test_acc=0.5729  test_auroc=0.5762
  layer=07  train_acc=0.9527  train_auroc=0.9912  test_acc=0.5830  test_auroc=0.5883
  layer=08  train_acc=0.9663  train_auroc=0.9953  test_acc=0.5756  test_auroc=0.5817
  layer=09  train_acc=0.9739  train_auroc=0.9972  test_acc=0.5841  test_auroc=0.5847
  layer=10  train_acc=0.9850  train_auroc=0.9983  test_acc=0.5889  test_auroc=0.5964
  layer=11  train

,layer,n_train,n_test,train_acc,train_auroc,test_acc,test_auroc
0,0,7542,1885,0.694643,0.762335,0.549602,0.562712
1,1,7542,1885,0.705781,0.791604,0.557029,0.569129
2,2,7542,1885,0.762928,0.861275,0.563926,0.572904
3,3,7542,1885,0.834659,0.926711,0.560212,0.557798
4,4,7542,1885,0.882790,0.959794,0.569761,0.582217
5,5,7542,1885,0.909971,0.973705,0.581432,0.579453
6,6,7542,1885,0.928666,0.984988,0.572944,0.576183
7,7,7542,1885,0.952665,0.991177,0.583024,0.588336
8,8,7542,1885,0.966322,0.995322,0.575597,0.581669
9,9,7542,1885,0.973880,0.997241,0.584085,0.584670


## Step 3: Domain Ablation
Train on one domain, test on another (or same) domain for each layer.

In [ ]:
def run_domain_ablation(features_dir, C):
    features_path = Path(features_dir)
    shared = np.load(features_path / "labels_domains_ids.npz", allow_pickle=True)
    y = shared["y"]
    domains = shared["domain"]

    unique_domains = list(dict.fromkeys(str(d) for d in domains))
    print(f"Domains found: {unique_domains}")

    layer_files = sorted(features_path.glob("layer_*.npz"), key=lambda p: int(p.stem.split("_")[1]))
    rows = []

    for layer_file in layer_files:
        layer_idx = int(layer_file.stem.split("_")[1])
        X = np.load(layer_file)["X"]

        for train_d in unique_domains:
            for test_d in unique_domains:
                train_mask = domains == train_d
                test_mask = domains == test_d

                clf = train_logreg_probe(X[train_mask], y[train_mask], C=C)
                in_m = eval_binary_classifier(clf, X[train_mask], y[train_mask])
                cross_m = eval_binary_classifier(clf, X[test_mask], y[test_mask])

                rows.append({
                    "layer": layer_idx, "train_domain": train_d, "test_domain": test_d,
                    "n_train": int(train_mask.sum()), "n_test": int(test_mask.sum()),
                    "in_acc": in_m["acc"], "in_auroc": in_m["auroc"],
                    "cross_acc": cross_m["acc"], "cross_auroc": cross_m["auroc"],
                })

        print(f"  layer={layer_idx:02d} done")

    return pd.DataFrame(rows).sort_values(["layer", "train_domain", "test_domain"]).reset_index(drop=True)

In [ ]:
ablation_df = run_domain_ablation(FEATURES_DIR, C=PROBE_C)
ablation_df.to_csv("domain_ablation_results.csv", index=False)
print("Saved domain_ablation_results.csv")
ablation_df

Domains found: ['boolq']
  layer=00 done
  layer=01 done
  layer=02 done
  layer=03 done
  layer=04 done
  layer=05 done
  layer=06 done
  layer=07 done
  layer=08 done
  layer=09 done
  layer=10 done
  layer=11 done
  layer=12 done
  layer=13 done


KeyboardInterrupt: 

## Step 4: Geometric Evaluation
PCA visualization and distance analysis on the best layer.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


def sample_pairs(a_idx, b_idx, n_pairs, seed):
    rng = np.random.RandomState(seed)
    pairs, seen = [], set()
    if len(a_idx) == 0 or len(b_idx) == 0:
        return pairs
    for _ in range(n_pairs * 10 + 100):
        if len(pairs) >= n_pairs:
            break
        i = int(a_idx[rng.randint(0, len(a_idx))])
        j = int(b_idx[rng.randint(0, len(b_idx))])
        if i == j:
            continue
        key = (min(i, j), max(i, j))
        if key not in seen:
            seen.add(key)
            pairs.append(key)
    return pairs


def cosine_distance(x, y):
    denom = float(np.linalg.norm(x) * np.linalg.norm(y))
    if denom <= 1e-12:
        return float("nan")
    return float(1.0 - np.dot(x, y) / denom)


def mahalanobis_distance(x, y, inv_cov):
    d = x - y
    v = float(np.dot(np.dot(d, inv_cov), d))
    return float(np.sqrt(max(v, 0.0)))


def summary_stats(values):
    arr = np.array(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return {"n": 0, "mean": float("nan"), "std": float("nan")}
    return {"n": len(arr), "mean": float(np.mean(arr)), "std": float(np.std(arr))}


def run_geometric_eval(features_dir, layer, n_pairs, seed, cov_reg):
    base = Path(features_dir)
    shared = np.load(base / "labels_domains_ids.npz", allow_pickle=True)
    y = shared["y"].astype(int)
    X = np.load(base / f"layer_{int(layer):02d}.npz")["X"]

    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]

    pairs00 = sample_pairs(idx0, idx0, n_pairs, seed + 1)
    pairs11 = sample_pairs(idx1, idx1, n_pairs, seed + 2)
    pairs01 = sample_pairs(idx0, idx1, n_pairs, seed + 3)

    cov = np.cov(X, rowvar=False) + cov_reg * np.eye(X.shape[1])
    inv_cov = np.linalg.pinv(cov)

    cos00 = [cosine_distance(X[i], X[j]) for i, j in pairs00]
    cos11 = [cosine_distance(X[i], X[j]) for i, j in pairs11]
    cos01 = [cosine_distance(X[i], X[j]) for i, j in pairs01]
    mah00 = [mahalanobis_distance(X[i], X[j], inv_cov) for i, j in pairs00]
    mah11 = [mahalanobis_distance(X[i], X[j], inv_cov) for i, j in pairs11]
    mah01 = [mahalanobis_distance(X[i], X[j], inv_cov) for i, j in pairs01]

    pca = PCA(n_components=2, random_state=seed)
    proj = pca.fit_transform(X)

    fig, ax = plt.subplots(figsize=(8, 6))
    m0 = y == 0
    m1 = y == 1
    ax.scatter(proj[m0, 0], proj[m0, 1], s=12, alpha=0.6, label="label=0 (false)")
    ax.scatter(proj[m1, 0], proj[m1, 1], s=12, alpha=0.6, label="label=1 (true)")
    ax.set_title(f"PCA Projection (Layer {int(layer):02d})")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.2f}% var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.2f}% var)")
    ax.legend(loc="best")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()

    print(f"\n--- Distance Summary (Layer {layer}) ---")
    print(f"Cosine  | within-0: {summary_stats(cos00)['mean']:.4f}  within-1: {summary_stats(cos11)['mean']:.4f}  between: {summary_stats(cos01)['mean']:.4f}")
    print(f"Mahal.  | within-0: {summary_stats(mah00)['mean']:.4f}  within-1: {summary_stats(mah11)['mean']:.4f}  between: {summary_stats(mah01)['mean']:.4f}")

    return {
        "layer": layer,
        "pca_explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
        "cosine": {"within_0": summary_stats(cos00), "within_1": summary_stats(cos11), "between": summary_stats(cos01)},
        "mahalanobis": {"within_0": summary_stats(mah00), "within_1": summary_stats(mah11), "between": summary_stats(mah01)},
    }

In [ ]:
print(f"Running geometric eval on best layer: {BEST_LAYER}")
geo_results = run_geometric_eval(FEATURES_DIR, layer=BEST_LAYER, n_pairs=N_PAIRS, seed=SEED, cov_reg=COV_REG)

## Step 5: Summary & Download
View final results and download outputs.

In [ ]:
print("=" * 60)
print("LAYER SWEEP SUMMARY")
print("=" * 60)
print(sweep_df.to_string(index=False))

print(f"\nBest layer: {BEST_LAYER}")
print(f"Best test AUROC: {best_row['test_auroc']:.4f}")
print(f"Best test accuracy: {best_row['test_acc']:.4f}")

majority_class_ratio = max(sweep_df.iloc[0]['n_train'], sweep_df.iloc[0]['n_test']) / (sweep_df.iloc[0]['n_train'] + sweep_df.iloc[0]['n_test'])
print(f"\nNote: Check label balance in your dataset. If imbalanced, AUROC is more reliable than accuracy.")

In [ ]:
import shutil

shutil.make_archive("features", "zip", ".", FEATURES_DIR)
print("Created features.zip")

from google.colab import files
files.download("layer_sweep_results.csv")
files.download("domain_ablation_results.csv")
files.download("features.zip")

# step 6
Check for Non-Linear features

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler


MLP_HIDDEN_SIZE = 32 # changed from 256
MLP_DROPOUT = 0.3
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_EPOCHS = 50
MLP_BATCH = 256


class _MLPNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPProbe:
    """Thin wrapper so eval_binary_classifier can call .predict_proba()."""

    def __init__(self, input_dim, hidden_dim=256, dropout=0.3, lr=1e-3,
                 weight_decay=1e-4, epochs=50, batch_size=256, seed=42, device="cpu"):
        torch.manual_seed(seed)
        self.device = torch.device(device)
        self.model = _MLPNet(input_dim, hidden_dim, dropout).to(self.device)
        self.lr = lr
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.batch_size = batch_size

    def fit(self, X, y):
        self.model.train()
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))

        n_pos = float(y.sum())
        n_neg = float(len(y) - n_pos)
        sample_weights = np.where(y == 1, 1.0 / n_pos, 1.0 / n_neg)
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(y), replacement=True)
        loader = DataLoader(ds, batch_size=self.batch_size, sampler=sampler)

        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.BCEWithLogitsLoss()

        for epoch in range(self.epochs):
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                opt.zero_grad()
                loss_fn(self.model(xb), yb).backward()
                opt.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            logits = self.model(torch.tensor(X, dtype=torch.float32).to(self.device))
            p1 = torch.sigmoid(logits).cpu().numpy()
        return np.column_stack([1 - p1, p1])


def run_mlp_sweep(features_dir, test_size, seed, hidden_size, dropout, device):
    features_path = Path(features_dir)
    shared = np.load(features_path / "labels_domains_ids.npz", allow_pickle=True)
    y = shared["y"]

    train_idx, test_idx = build_train_test_indices(len(y), test_size, seed)

    layer_files = sorted(features_path.glob("layer_*.npz"), key=lambda p: int(p.stem.split("_")[1]))
    print(f"MLP Probe: {len(layer_files)} layers, hidden={hidden_size}, dropout={dropout}, device={device}")
    print(f"  n_train={len(train_idx)}, n_test={len(test_idx)}")

    rows = []
    for layer_file in layer_files:
        layer_idx = int(layer_file.stem.split("_")[1])
        X = np.load(layer_file)["X"]

        clf = MLPProbe(
            input_dim=X.shape[1], hidden_dim=hidden_size, dropout=dropout,
            lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY, epochs=MLP_EPOCHS,
            batch_size=MLP_BATCH, seed=seed, device=device,
        ).fit(X[train_idx], y[train_idx])

        train_m = eval_binary_classifier(clf, X[train_idx], y[train_idx])
        test_m = eval_binary_classifier(clf, X[test_idx], y[test_idx])

        row = {
            "layer": layer_idx, "n_train": len(train_idx), "n_test": len(test_idx),
            "train_acc": train_m["acc"], "train_auroc": train_m["auroc"],
            "test_acc": test_m["acc"], "test_auroc": test_m["auroc"],
        }
        rows.append(row)
        print(f"  layer={layer_idx:02d}  train_acc={row['train_acc']:.4f}  train_auroc={row['train_auroc']:.4f}  test_acc={row['test_acc']:.4f}  test_auroc={row['test_auroc']:.4f}")

    return pd.DataFrame(rows).sort_values("layer").reset_index(drop=True)

In [ ]:
mlp_df = run_mlp_sweep(FEATURES_DIR, test_size=TEST_SIZE, seed=SEED, hidden_size=MLP_HIDDEN_SIZE, dropout=MLP_DROPOUT, device=DEVICE)

mlp_best = mlp_df.loc[mlp_df["test_auroc"].idxmax()]
print(f"\nMLP best layer: {int(mlp_best['layer'])} (test_auroc={mlp_best['test_auroc']:.4f})")
print(f"LR  best layer: {BEST_LAYER} (test_auroc={best_row['test_auroc']:.4f})")
print(f"MLP improvement over LR: {mlp_best['test_auroc'] - best_row['test_auroc']:+.4f}")

mlp_df.to_csv("mlp_sweep_results_enhanced_overfitting_measures.csv", index=False)
print("Saved mlp_sweep_results_enhanced_overfitting_measures.csv")
mlp_df

MLP Probe: 41 layers, hidden=256, dropout=0.3, device=cuda
  n_train=7542, n_test=1885
  layer=00  train_acc=0.6928  train_auroc=0.7873  test_acc=0.5655  test_auroc=0.5556
  layer=01  train_acc=0.7690  train_auroc=0.8860  test_acc=0.5496  test_auroc=0.5736
  layer=02  train_acc=0.8457  train_auroc=0.9456  test_acc=0.5671  test_auroc=0.5745
  layer=03  train_acc=0.8937  train_auroc=0.9723  test_acc=0.5629  test_auroc=0.5647
  layer=04  train_acc=0.9427  train_auroc=0.9901  test_acc=0.5851  test_auroc=0.5899
  layer=05  train_acc=0.9483  train_auroc=0.9923  test_acc=0.5989  test_auroc=0.6051
  layer=06  train_acc=0.9643  train_auroc=0.9964  test_acc=0.5889  test_auroc=0.6026
  layer=07  train_acc=0.9752  train_auroc=0.9968  test_acc=0.6127  test_auroc=0.6209
  layer=08  train_acc=0.9822  train_auroc=0.9974  test_acc=0.6207  test_auroc=0.6321
  layer=09  train_acc=0.9869  train_auroc=0.9989  test_acc=0.6186  test_auroc=0.6325
  layer=10  train_acc=0.9909  train_auroc=0.9995  test_acc=0.62

,layer,n_train,n_test,train_acc,train_auroc,test_acc,test_auroc
0,0,7542,1885,0.692787,0.787288,0.565517,0.555630
1,1,7542,1885,0.769027,0.885972,0.549602,0.573591
2,2,7542,1885,0.845664,0.945646,0.567109,0.574522
3,3,7542,1885,0.893662,0.972349,0.562865,0.564666
4,4,7542,1885,0.942721,0.990130,0.585146,0.589874
5,5,7542,1885,0.948290,0.992328,0.598939,0.605112
6,6,7542,1885,0.964333,0.996379,0.588859,0.602557
7,7,7542,1885,0.975206,0.996777,0.612732,0.620894
8,8,7542,1885,0.982233,0.997356,0.620690,0.632143
9,9,7542,1885,0.986874,0.998863,0.618568,0.632520


# Data Collection Pipeline

As of 3/10/2026 at 7PM recall:

1: We want to run experiments that check if the model's predicted truth value matches the true truth value

2: Afterwards we are considering grabbing the model's activations to investigate its internal state during various true/false tasks

The above pipeline combined a lot of disjoint steps into functions as a design choice because we only wanted to examine BoolQ. Now we may want to look at arbitrary T/F datasets, so I have written some helper functions


In [ ]:
# This function is essentially just a wrapper around the logic of getting a row/column in parquet/csv files. It will be called so
# much below that I wrote a helper function for it
def extract_dataset_feature(dataset_path : str, statement_label,dataset_type="csv"):
    # I saw we were using parquet files but most of the datasets we want to use are in csv format. Since it's a one-time read with data being moved to other formats,
    # I figure it's fine to use either so there is an option here to use csv or parquet
    dataset = ""
    if dataset_type == "csv":
      dataset = pd.read_csv(dataset_path)
    elif dataset_type == "parquet":
      dataset = pd.read_parquet(dataset_path)
    else:
      raise ValueError(f"Unsupported dataset type: {dataset_type}")

    statements = dataset[statement_label]
    return statements.tolist()

# Example testing with BoolQ: It works
# print(extract_dataset_feature(DATASET_PATH,"statement",dataset_type="parquet"))



['do iran and afghanistan speak the same language', 'do good samaritan laws protect those who help at an accident', 'is windows movie maker part of windows essentials', 'is confectionary sugar the same as powdered sugar', 'is elder scrolls online the same as skyrim', 'can you use oyster card at epsom station', "will there be a season 4 of da vinci's demons", 'is the federal court the same as the supreme court', 'did abraham lincoln write the letter in saving private ryan', 'is batman and robin a sequel to batman forever', 'is a wolverine the same as a badger', 'will there be a green lantern 2 movie', 'does the icc has jurisdiction in the united states', 'calcium carbide cac2 is the raw material for the production of acetylene', 'is there a now you see me 3 coming out', 'does a penalty shoot out goal count towards the golden boot', 'will there be a new season of cutthroat kitchen', 'is jack sikma in the hall of fame', 'can elves and humans mate lord of the rings', 'the boy in the plasti

In [ ]:
def model_output(model_name, dataset_path, out_dir, batch_size, max_length, device,
                 prompt_template, use_4bit=False):
    """Generate the model's greedy next-token for each prompt and classify as true/false.

    Saves a CSV with columns: prompt_num, answer, output, hallucination
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    df = pd.read_parquet(dataset_path)
    required = {"id", "domain", "label", "statement"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {', '.join(sorted(missing))}")

    texts = [prompt_template.format(statement=str(s)) for s in df["statement"].tolist()]
    ground_truth = df["label"].astype(int).to_numpy()
    print(f"Loaded {len(texts)} prompts")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    dtype = torch.float16 if device.startswith("cuda") else torch.float32

    if use_4bit and device.startswith("cuda"):
        print("Loading model with 4-bit quantization (NF4)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto",
        )
    elif device.startswith("cuda"):
        print(f"Loading model in {dtype}...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=dtype, device_map="auto",
        )
    else:
        print(f"Loading model in {dtype}...")
        model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype)
        model.to(device)
    model.eval()
    input_device = _infer_input_device(model, fallback_device=device)
    print(f"Model loaded. Input device: {input_device}")

    results = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for batch_num, i in enumerate(range(0, len(texts), batch_size)):
        if batch_num % 20 == 0:
            print(f"  Batch {batch_num + 1}/{total_batches}")
        batch_texts = texts[i : i + batch_size]

        enc = tokenizer(
            batch_texts, return_tensors="pt", padding=True,
            truncation=True, max_length=max_length,
        )
        enc = {k: v.to(input_device) for k, v in enc.items()}

        with torch.no_grad():
            generated_ids = model.generate(
                **enc,
                max_new_tokens=1,
                do_sample=False,
            )

        new_token_ids = generated_ids[:, enc["input_ids"].shape[1]:]
        decoded_tokens = tokenizer.batch_decode(new_token_ids, skip_special_tokens=True)

        for j, raw_output in enumerate(decoded_tokens):
            idx = i + j
            output_stripped = raw_output.strip()
            output_lower = output_stripped.lower()

            if output_lower == "true":
                model_answer = 1
            elif output_lower == "false":
                model_answer = 0
            else:
                model_answer = -1

            gt = int(ground_truth[idx])
            gt_str = "True" if gt == 1 else "False"

            if model_answer == -1:
                hallucinated = "N/A"
            else:
                hallucinated = "Yes" if model_answer != gt else "No"

            results.append({
                "prompt_num": idx,
                "answer": gt_str,
                "output": output_stripped,
                "hallucination": hallucinated,
            })

    results_df = pd.DataFrame(results)
    csv_path = out_path / "model_outputs.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"\nSaved {len(results_df)} results to {csv_path}")

    valid = results_df[results_df["hallucination"] != "N/A"]
    if len(valid) > 0:
        n_hall = (valid["hallucination"] == "Yes").sum()
        print(f"Hallucination rate: {n_hall}/{len(valid)} ({100 * n_hall / len(valid):.1f}%)")
    n_na = (results_df["hallucination"] == "N/A").sum()
    if n_na > 0:
        print(f"Unrecognized outputs (not true/false): {n_na}")
        print("Sample unrecognized:", results_df[results_df["hallucination"] == "N/A"]["output"].value_counts().head(10).to_dict())

    return results_df

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 152)

## Step 7: Model Output & Hallucination Detection
Use `model.generate` to get the model's greedy next-token prediction for each prompt, classify it as True/False, and flag hallucinations where the model's answer disagrees with ground truth.

In [ ]:
OUTPUT_DIR = "model_outputs"
PROMPT_TEMPLATE_TF = "True or False: {statement}\nAnswer:"

output_df = model_output(
    model_name=MODEL_NAME,
    dataset_path=DATASET_PATH,
    out_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE,
    prompt_template=PROMPT_TEMPLATE_TF,
    use_4bit=USE_4BIT,
)

print("\n--- First 20 rows ---")
print(output_df.head(20).to_string(index=False))
print(f"\n--- Output value counts ---")
print(output_df["output"].value_counts().head(15))
print(f"\n--- Hallucination breakdown ---")
print(output_df["hallucination"].value_counts())

from google.colab import files
files.download(f"{OUTPUT_DIR}/model_outputs.csv")